In [96]:
import pandas as pd
import logging
#from sqlalchemy import create_engine
import os
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [89]:
file_path = '~/Downloads/example_sales_data.csv'
df_raw = pd.read_csv(file_path)
print(df_raw.head())

         date   store ID          PRODUCT_NAME   quantity     Price  \
0  2023-01-15  Store_005       Laptop Ultra X1        2.0  $1299.99   
1   1/16/2023  STORE_005     Desktop PC Gaming        1.0   1499.99   
2  2023/01/17  Store_005  Wireless Headphones         3.0    $89.99   
3  01-18-2023  store_005        Smartphone X12        NaN    799.99   
4  2023-01-19  Store_005            Tablet Pro        0.0    649.99   

  customer_type Payment Method Transaction_ID  discount% region  
0       Regular            NaN      TRX-45892         0%   West  
1       Premium    Credit Card      TRX-45893       0.05   West  
2       Regular    Credit Card      TRX-45894          0   West  
3       Premier          Debit      TRX-45895        10%   west  
4       Regular           Cash      TRX-45896        NaN   WEST  


In [90]:
print(df_raw.shape)
print(df_raw.columns)
print(df_raw.dtypes)
print(df_raw.describe(include='all'))

(51, 10)
Index(['date', 'store ID', 'PRODUCT_NAME', ' quantity', 'Price',
       'customer_type', 'Payment Method', 'Transaction_ID', ' discount%',
       'region'],
      dtype='str')
date                  str
store ID              str
PRODUCT_NAME          str
 quantity         float64
Price                 str
customer_type         str
Payment Method        str
Transaction_ID        str
 discount%            str
region                str
dtype: object
              date   store ID     PRODUCT_NAME   quantity   Price  \
count           51         51               51  46.000000      51   
unique          51         21               18        NaN      31   
top     2023-01-15  Store_005  Laptop Ultra X1        NaN  799.99   
freq             1          7                3        NaN       3   
mean           NaN        NaN              NaN   2.565217     NaN   
std            NaN        NaN              NaN   2.587596     NaN   
min            NaN        NaN              NaN   0.000000 

In [13]:
print(df_raw.duplicated().sum())

0


In [91]:
#df_raw.columns = [col.strip().lower().replace(' ', '_') for col in df_raw.columns]
df_raw.columns = [col.strip().lower().replace(' ', '_').replace('%', '_pct') for col in df_raw.columns]
print(df_raw.head())

         date   store_id          product_name  quantity     price  \
0  2023-01-15  Store_005       Laptop Ultra X1       2.0  $1299.99   
1   1/16/2023  STORE_005     Desktop PC Gaming       1.0   1499.99   
2  2023/01/17  Store_005  Wireless Headphones        3.0    $89.99   
3  01-18-2023  store_005        Smartphone X12       NaN    799.99   
4  2023-01-19  Store_005            Tablet Pro       0.0    649.99   

  customer_type payment_method transaction_id discount_pct region  
0       Regular            NaN      TRX-45892           0%   West  
1       Premium    Credit Card      TRX-45893         0.05   West  
2       Regular    Credit Card      TRX-45894            0   West  
3       Premier          Debit      TRX-45895          10%   west  
4       Regular           Cash      TRX-45896          NaN   WEST  


In [26]:
df_raw['sale_date'] = pd.to_datetime(
    df_raw['date'],
    format='mixed',
    errors='coerce',          # Turn unparseable into NaT
    dayfirst=False            # Set True if your data is mostly DD/MM
)

In [27]:
df_raw

,date,store_id,product_name,quantity,price,customer_type,payment_method,transaction_id,discount%,region,sale_date
0,2023-01-15,Store_005,Laptop Ultra X1,2.0,$1299.99,Regular,NaN,TRX-45892,0%,West,2023-01-15
1,1/16/2023,STORE_005,Desktop PC Gaming,1.0,1499.99,Premium,Credit Card,TRX-45893,0.05,West,2023-01-16
2,2023/01/17,Store_005,Wireless Headphones,3.0,$89.99,Regular,Credit Card,TRX-45894,0,West,2023-01-17
3,01-18-2023,store_005,Smartphone X12,NaN,799.99,Premier,Debit,TRX-45895,10%,west,2023-01-18
4,2023-01-19,Store_005,Tablet Pro,0.0,649.99,Regular,Cash,TRX-45896,NaN,WEST,2023-01-19
5,01/20/23,Store_005,USB-C Cable 2m,5.0,$19.99,Regular,Credit Card,NaN,0%,West,2023-01-20
6,2023-01-21,StoreID_005,Wireless Mouse,2.0,$49.99,Regular,Debit Card,TRX-45898,0.00,West,2023-01-21
7,22-01-2023,Store-005,External SSD 1TB,1.0,159.99,Premium,NaN,TRX-45899,5,West,2023-01-22
8,2023-01-23,Store 005,Laptop Stand,4.0,$29.99,Regular,Credit Card,TRX-45900,0,west,2023-01-23
9,1/24/2023,STORE_005,Wireless Keyboard,1.0,79.99,Premier,Credit Card,TRX-45901,15%,West,2023-01-24


In [28]:
date_formats = [
    '%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%Y/%m/%d',
    '%d-%m-%Y', '%d.%m.%Y', '%d-%b-%Y', '%b-%d-%Y',
    '%d-%b-%y', '%b-%d-%y', '%Y%m%d', '%m-%d-%Y', '%m/%d/%y'
]

def parse_date(x):
    if pd.isna(x):                    # Handle NaN / None early
        return pd.NaT

    x = str(x).strip()
    if x.lower() in ['nan', 'null', '', 'none', 'n/a']:
        return pd.NaT

    for fmt in date_formats:
        try:
            # Return full datetime (best practice)
            return pd.to_datetime(x, format=fmt)
        except:
            continue

    logging.warning(f"Could not parse date: {x}")
    return pd.NaT

In [29]:
df_raw['sale_date'] = df_raw['date'].apply(parse_date)

In [32]:
# 2. Store ID normalization
df_raw['store_id'] = df_raw['store_id'].str.strip().str.upper().str.replace(r'[^0-9]', '', regex=True)

In [34]:
df_raw['store_id'] = df_raw['store_id'].astype('Int64')
df_raw['store_id']

0     5
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
11    5
12    5
13    5
14    5
15    5
16    5
17    6
18    6
19    6
20    6
21    6
22    6
23    6
24    6
25    6
26    6
27    6
28    6
29    6
30    6
31    6
32    6
33    6
34    7
35    7
36    7
37    7
38    7
39    7
40    7
41    7
42    7
43    7
44    7
45    7
46    7
47    7
48    7
49    7
50    7
Name: store_id, dtype: Int64

In [36]:
df_raw['product_name'].unique()

<ArrowStringArray>
[     'Laptop Ultra X1',    'Desktop PC Gaming', 'Wireless Headphones ',
       'Smartphone X12',           'Tablet Pro',       'USB-C Cable 2m',
       'Wireless Mouse',     'External SSD 1TB',         'Laptop Stand',
    'Wireless Keyboard',  'Power Bank 20000mAh',    'Bluetooth Speaker',
     'Monitor, 27-inch',           'HDMI Cable',     'Wireless Charger',
          'Airpods Pro', 'Smart Watch Series 7',  'Wireless Headphones']
Length: 18, dtype: str

In [37]:
df_raw['product_name'] = df_raw['product_name'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [38]:
df_raw['product_name'].unique()

<ArrowStringArray>
[     'Laptop Ultra X1',    'Desktop PC Gaming',  'Wireless Headphones',
       'Smartphone X12',           'Tablet Pro',       'USB-C Cable 2m',
       'Wireless Mouse',     'External SSD 1TB',         'Laptop Stand',
    'Wireless Keyboard',  'Power Bank 20000mAh',    'Bluetooth Speaker',
     'Monitor, 27-inch',           'HDMI Cable',     'Wireless Charger',
          'Airpods Pro', 'Smart Watch Series 7']
Length: 17, dtype: str

In [40]:
# 4. Quantity
df_raw['quantity'] = pd.to_numeric(df_raw['quantity'], errors='coerce')
df_raw['quantity'] = df_raw['quantity'].fillna(0).astype(int)

In [41]:
df_raw['quantity']

0      2
1      1
2      3
3      0
4      0
5      5
6      2
7      1
8      4
9      1
10     2
11     1
12     1
13     3
14     2
15     1
16     0
17     1
18     2
19     5
20     1
21     2
22    10
23     3
24     1
25     2
26     4
27     1
28     0
29     1
30     5
31     3
32     1
33     2
34     3
35     1
36     4
37     2
38     1
39    15
40     2
41     0
42     3
43     1
44     2
45     1
46     1
47     6
48     0
49     2
50     1
Name: quantity, dtype: int64

In [43]:
df_raw['price'].unique()

<ArrowStringArray>
['$1299.99',  '1499.99',  ' $89.99',   '799.99',   '649.99',   '$19.99',
   '$49.99',   '159.99',   '$29.99',    '79.99',   '$59.99',   '129.99',
   '349.99',   '$14.99',    '39.99',       'na',   '299.99',  '1299.99',
 '$1499.99',    '89.99',  '$649.99',    '19.99',    '29.99',   '$79.99',
    '59.99',  '$349.99',    '14.99',   '249.99',  '$299.99',   ' 89.99',
    'Error']
Length: 31, dtype: str

In [45]:
# 5. Price
df_raw['price'] = df_raw['price'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip()
df_raw['price'] = pd.to_numeric(df_raw['price'], errors='coerce')
df_raw['price'].unique()

array([1299.99, 1499.99,   89.99,  799.99,  649.99,   19.99,   49.99,
        159.99,   29.99,   79.99,   59.99,  129.99,  349.99,   14.99,
         39.99,     nan,  299.99,  249.99])

In [47]:
df_raw['discount%'].unique()

<ArrowStringArray>
['0%', '0.05', '0', '10%', nan, '0.00', '5', '15%', '5%', '10', '10.0', '15']
Length: 12, dtype: str

In [ ]:
# 6. Discount
df_raw['discount%'] = df_raw[' discount%'].astype(str).str.replace(r'[%]', '', regex=True).str.strip()
df_raw['discount%'] = pd.to_numeric(df_raw['discount%'], errors='coerce').fillna(0)

In [49]:
import numpy as np

# 6. Normalize Discount Column
def normalize_discount(x):
    if pd.isna(x) or str(x).strip().lower() in ['nan', 'none', '']:
        return 0.0

    x = str(x).strip().replace('%', '').strip()   # Remove % sign

    try:
        val = float(x)

        # If it's a decimal proportion (0 to 1), convert to percentage
        if 0 <= val <= 1:
            val = val * 100

        # Cap at reasonable values (optional but recommended)
        if val < 0:
            return 0.0
        if val > 100:
            return 100.0

        return round(val, 4)   # Keep up to 4 decimals if needed

    except:
        return 0.0


# Apply the function
df_raw['discount_pct'] = df_raw['discount%'].apply(normalize_discount)

In [51]:
df_raw['discount_pct']

0      0.0
1      5.0
2      0.0
3     10.0
4      0.0
5      0.0
6      0.0
7      5.0
8      0.0
9     15.0
10     0.0
11     0.0
12    10.0
13     0.0
14     0.0
15     0.0
16     5.0
17     0.0
18    10.0
19     0.0
20     0.0
21     5.0
22     0.0
23    10.0
24     0.0
25     0.0
26     5.0
27     0.0
28     0.0
29     0.0
30     0.0
31    10.0
32     0.0
33     5.0
34    15.0
35     0.0
36     0.0
37     5.0
38     0.0
39     0.0
40     5.0
41     0.0
42     0.0
43    10.0
44     0.0
45     0.0
46     5.0
47     0.0
48     0.0
49     0.0
50    15.0
Name: discount_pct, dtype: float64

In [53]:
df_raw['customer_type'].str.strip().str.title()

0             Regular
1             Premium
2             Regular
3             Premier
4             Regular
5             Regular
6             Regular
7             Premium
8             Regular
9             Premier
10            Regular
11            Regular
12            Premium
13            Regular
14            Regular
15            Premium
16    Regularcustomer
17            Premium
18            Regular
19            Regular
20            Premium
21            Regular
22            Regular
23            Premium
24            Regular
25            Regular
26            Premier
27            Regular
28            Premium
29            Regular
30            Regular
31            Premier
32            Regular
33            Premium
34            Regular
35            Premium
36            Regular
37            Premier
38            Regular
39            Regular
40            Premium
41            Regular
42            Regular
43            Premier
44            Regular
45        

In [75]:
df_raw['region'].unique()

<ArrowStringArray>
['West', 'North West', 'East', 'North East', 'South', 'South East']
Length: 6, dtype: str

In [79]:
# 7. Other fields
df_raw['customer_type'] = df_raw['customer_type'].str.strip().replace('customer', '').str.title().str.replace('Regularcustomer', 'Regular', regex=False).str.replace('Premier', 'Premium', regex=False)
df_raw['payment_method'] = df_raw['payment_method'].fillna('Unknown').str.strip().str.title()
df_raw['transaction_id'] = df_raw['transaction_id'].fillna('UNKNOWN').str.strip().str.upper()
df_raw['region'] = df_raw['region'].str.strip().str.title()

In [73]:
# 7. Customer Type Standardization
def clean_customer_type(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().title()

    # Apply your rules
    if 'Reg' in x:
        return 'Regular'
    elif 'Pre' in x:
        return 'Premier'

    # Fallback (in case something doesn't match)
    return x


# Apply the function
df_raw['customer_type'] = df_raw['customer_type'].apply(clean_customer_type)

In [82]:
# 8. Derived columns
df_raw['total_amount'] = df_raw['quantity'] * df_raw['price'] * (1 - df_raw['discount_pct']/100)
df_raw['loaded_at'] = datetime.utcnow()

In [97]:
df_raw['loaded_at'] = datetime.now(timezone.utc)

In [85]:

df_raw['total_amount'] = df_raw['total_amount'].fillna(0)
df_raw['total_amount']=df_raw['total_amount'].round(2)
df_raw

,date,store_id,product_name,quantity,price,customer_type,payment_method,transaction_id,discount%,region,sale_date,discount_pct,total_amount,loaded_at
0,2023-01-15,5,Laptop Ultra X1,2,1299.99,Regular,Unknown,TRX-45892,0%,West,2023-01-15,0.0,2599.98,2026-05-07 00:03:12.271195
1,1/16/2023,5,Desktop PC Gaming,1,1499.99,Premium,Credit Card,TRX-45893,0.05,West,2023-01-16,5.0,1424.99,2026-05-07 00:03:12.271195
2,2023/01/17,5,Wireless Headphones,3,89.99,Regular,Credit Card,TRX-45894,0,West,2023-01-17,0.0,269.97,2026-05-07 00:03:12.271195
3,01-18-2023,5,Smartphone X12,0,799.99,Premium,Debit,TRX-45895,10%,West,2023-01-18,10.0,0.00,2026-05-07 00:03:12.271195
4,2023-01-19,5,Tablet Pro,0,649.99,Regular,Cash,TRX-45896,NaN,West,2023-01-19,0.0,0.00,2026-05-07 00:03:12.271195
5,01/20/23,5,USB-C Cable 2m,5,19.99,Regular,Credit Card,UNKNOWN,0%,West,2023-01-20,0.0,99.95,2026-05-07 00:03:12.271195
6,2023-01-21,5,Wireless Mouse,2,49.99,Regular,Debit Card,TRX-45898,0.00,West,2023-01-21,0.0,99.98,2026-05-07 00:03:12.271195
7,22-01-2023,5,External SSD 1TB,1,159.99,Premium,Unknown,TRX-45899,5,West,2023-01-22,5.0,151.99,2026-05-07 00:03:12.271195
8,2023-01-23,5,Laptop Stand,4,29.99,Regular,Credit Card,TRX-45900,0,West,2023-01-23,0.0,119.96,2026-05-07 00:03:12.271195
9,1/24/2023,5,Wireless Keyboard,1,79.99,Premium,Credit Card,TRX-45901,15%,West,2023-01-24,15.0,67.99,2026-05-07 00:03:12.271195


In [86]:
explain:

SyntaxError: invalid syntax (2927112033.py, line 1)

In [ ]:
def load_to_postgres(df: pd.DataFrame):
    conn_str = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    engine = create_engine(conn_str)

    try:
        with engine.connect() as conn:
            conn.execute("CREATE SCHEMA IF NOT EXISTS raw;")
            conn.execute("CREATE SCHEMA IF NOT EXISTS analytics;")

        df.to_sql('sales_raw', engine, schema='raw', if_exists='replace', index=False)
        logging.info(f"✅ Loaded {len(df)} cleaned rows into raw.sales_raw")

        # Validation
        with engine.connect() as conn:
            count = conn.execute("SELECT COUNT(*) FROM raw.sales_raw").scalar()
            logging.info(f"Validation: {count} rows loaded")

            nulls = conn.execute("""
                SELECT COUNT(*) FROM raw.sales_raw
                WHERE sale_date IS NULL OR price IS NULL OR quantity < 0
            """).scalar()
            if nulls > 0:
                logging.warning(f"⚠️ Found {nulls} rows with critical nulls")

    except Exception as e:
        logging.error(f"❌ Load failed: {e}")
        raise

